### Taxi Metrics

In [0]:
from pyspark.sql import functions as F

df_silver_taxi = spark.read.table("unified_lakehouse.silver.taxi")

gold_taxi_metrics_df = (
    df_silver_taxi
    .groupBy("PULocationID", "description")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue")
    )
)

gold_taxi_metrics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .clusterBy("PULocationID", "description") \
    .saveAsTable("unified_lakehouse.gold.taxi_metrics")

### Driver Metrics

In [0]:
from pyspark.sql import functions as F

df_silver_drivers = spark.read.table("unified_lakehouse.silver.drivers")

gold_driver_metrics_df = (
    df_silver_drivers
    .groupBy("experience")
    .agg(
        F.count("*").alias("drivers_count"),
        F.round(F.mean("rating"), 2).alias("avg_rating")
    )
)

gold_driver_metrics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("unified_lakehouse.gold.driver_metrics")

spark.sql("OPTIMIZE unified_lakehouse.gold.driver_metrics ZORDER BY (experience)")

### User Metrics

In [0]:
from pyspark.sql import functions as F

df_silver_users = spark.read.table("unified_lakehouse.silver.users")

gold_user_metrics_df = (
    df_silver_users
    .groupBy("registration_date")
    .agg(F.count("*").alias("total_users"))
    .orderBy(F.col("total_users").desc())
)

gold_user_metrics_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("unified_lakehouse.gold.user_metrics")

### Maintenance

In [0]:
%sql
OPTIMIZE unified_lakehouse.gold.taxi_metrics;
OPTIMIZE unified_lakehouse.gold.driver_metrics;
OPTIMIZE unified_lakehouse.gold.user_metrics;